# 支线 11：LoRA — 怎么用一张 4090 微调 70B 模型？

> **背景**：你有一个 70B 参数的 LLM，想在 1000 条对话数据上做微调。
> 如果用全量微调，你需要 ~140GB 显存存模型 + ~280GB 存优化器状态 = 至少 8 张 A100。
> LoRA 说：不需要，一张 4090 就够了。
>
> 本 Part 目标：**从零理解 LoRA 的核心原理——矩阵低秩分解，然后手写一个 LoraLinear 接入 MiniGPT，证明它真的能学。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 1. 全量微调贵在哪？—— 一张表说清楚

先算一笔账。假设你要微调一个模型，单卡显存被以下东西占着：

| 组件 | 占用公式 | LLaMA-7B | LLaMA-70B |
|------|---------|----------|----------|
| 模型权重 (FP16) | `2 × N_bytes` | 14 GB | 140 GB |
| 优化器状态 (Adam) | `12 × N_bytes` | 84 GB | 840 GB |
| 梯度 | `2 × N_bytes` | 14 GB | 140 GB |
| 激活值 | 取决于 batch | ~8 GB | ~20 GB |
| **总计（全量微调）** | | **~120 GB** | **~1140 GB** |
| **总计（LoRA，r=16）** | 只存 LoRA 参数的优化器状态 | **~15 GB** | **~145 GB** |

核心：LoRA **不更新原始权重**。原始权重冻结，只训练旁路的小矩阵。

### 2. 核心直觉：为什么权重更新是「低秩」的？

**类比**：你有一幅画，想把它改成另一幅画。
- **全量微调** = 重新画整幅画（改动每一笔）
- **LoRA** = 在原画上覆盖一张半透明的薄纸，只改几个关键色调（改动是低秩的）

数学直觉：
> 一个预训练好的模型已经学到了通用的语言模式。
> 适应新任务时，你不需要推翻这些模式，只需要沿着少数几个方向「微调」它们。
> 这些方向的数量 << 原始权重的维度 → 更新矩阵 $\Delta W$ 是低秩的。

**低秩是什么意思？**

一个 `d × d` 的矩阵（d=4096），如果秩也是 4096，它有 1600 万个独立参数。

但低秩矩阵（秩=16）可以分解为 A × B：
```
ΔW (4096 × 4096) = A (4096 × 16) × B (16 × 4096)
参数从 16M → 13万（只有原来的 1/128！）
```

本质是：虽然 $\Delta W$ 是一个很大尺寸的矩阵，但它的「信息量」很小——
所有列都可以被 16 个基向量线性表示。

### 3. 手算拆解：LoRA 的前向传播到底在算什么？

记住核心公式：

$$h = Wx + \frac{\alpha}{r} \cdot A B x$$

其中：
- $W$：原始权重（冻结，不动！）
- $A$：`(d_out, r)` 的小矩阵（随机初始化，可训练）
- $B$：`(r, d_in)` 的小矩阵（初始化为 0，可训练）
- $\alpha$：缩放因子，通常设 `alpha = r` 的几倍
- $r$：秩，通常 `r ∈ {8, 16, 32, 64}`

**为什么 B 初始化为 0？** 这样训练开始时 $\Delta W = A B = 0$，模型行为 = 原始模型。
微调是**逐步**引入变化的，不会一上来就翻车。

下面用一个极小的例子手算：d_in=4, d_out=4, r=2

In [ ]:
# ============================================================
# 手算 LoRA：d_in=4, d_out=4, r=2
# ============================================================
torch.manual_seed(42)

d_in, d_out, r = 4, 4, 2
alpha = 2  # 缩放因子

# 原始权重（预训练好的，冻结）
W = torch.randn(d_out, d_in)
print("原始权重 W (4x4):")
print(W)
print(f"  参数数: {W.numel()}")

# LoRA 的两个小矩阵
A = torch.randn(d_out, r) * 0.02   # 用较小随机初始化
B = torch.zeros(r, d_in)            # ← 关键：B 初始化为 0！
print(f"\nA (d_out x r = 4x2):")
print(A)
print(f"B (r x d_in = 2x4)，初始化为 0:")
print(B)
print(f"  A 参数: {A.numel()}, B 参数: {B.numel()}, 合计: {A.numel() + B.numel()}")
print(f"  压缩比: {W.numel() / (A.numel() + B.numel()):.1f}x")

# 输入向量 x
x = torch.tensor([1.0, 0.5, -0.5, -1.0])
print(f"\n输入 x (4,): {x}")

# ============================================================
# 前向传播
# ============================================================
# 原始输出（没有 LoRA）
h_original = W @ x
print(f"\n原始输出 Wx: {h_original}")

# LoRA 旁路
h_lora = B @ x     # Step 1: (r, d_in) @ (d_in,) → (r,)    降维
h_lora = A @ h_lora  # Step 2: (d_out, r) @ (r,)  → (d_out,)  升维
scaling = alpha / r
delta = h_lora * scaling
print(f"\nLoRA 旁路计算:")
print(f"  Step 1 - Bx (r={r},): {h_lora}")
print(f"  Step 2 - A(Bx) (d_out={d_out},): {h_lora}")
print(f"  缩放因子 alpha/r: {scaling}")
print(f"  delta = (alpha/r) * A(Bx): {delta}")

# ★ 最终输出：原输出 + LoRA 修正
h_final = h_original + delta
print(f"\n最终输出 = Wx + (alpha/r)*ABx: {h_final}")

# ★ 验证：B=0 时，delta=0，输出 = 原始输出
print(f"\n验证 B=0 时 delta 全为 0: {(h_final == h_original).all().item()}")
print("(所以训练开始时，模型行为完全不变！)")

### 4. 代码实现：LoraLinear — 包装 nn.Linear

我们写一个 `LoraLinear` 类，使用方式和 `nn.Linear` 一样，但内部多了 LoRA 旁路。

In [ ]:
# ============================================================
# LoraLinear：包装 nn.Linear，加 LoRA 旁路
# ============================================================
class LoraLinear(nn.Module):
    """
    等价于: y = Wx + (alpha/r) * A(Bx)
    - W: 原始权重（requires_grad=False，冻结）
    - A: (out_features, r) 小矩阵（可训练）
    - B: (r, in_features)  小矩阵（可训练）
    """
    def __init__(self, linear: nn.Linear, r: int, alpha: int = None):
        super().__init__()
        # 保留原始层的引用
        self.linear = linear
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.r = r
        self.alpha = alpha if alpha is not None else r

        # 冻结原始权重
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # LoRA 参数：A 随机初始化，B 初始化为 0
        # A: (out_features, r)  B: (r, in_features)
        self.lora_A = nn.Parameter(torch.randn(self.out_features, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, self.in_features))
        self.scaling = self.alpha / self.r

    def forward(self, x):
        # Step 1: 原始输出（冻结部分，无梯度）
        original = self.linear(x)  # x @ W^T + bias
        # Step 2: LoRA 旁路 → dropout 稳定训练，r 很小时可省略
        #         数学上: x @ (lora_A @ lora_B)^T * scaling，即 (B^T @ A^T @ x^T)^T
        #         等价于: (x @ lora_B^T) @ lora_A^T * scaling
        lora_out = (x @ self.lora_B.T) @ self.lora_A.T * self.scaling
        return original + lora_out

    @property
    def weight(self):
        """合并后的等效权重: W + (alpha/r) * AB"""
        with torch.no_grad():
            delta_W = self.lora_A @ self.lora_B * self.scaling
            return self.linear.weight + delta_W


# 创建测试
torch.manual_seed(42)
linear = nn.Linear(4, 4, bias=False)
linear.weight.data = torch.randn(4, 4)  # 手动赋值权重

lora_layer = LoraLinear(linear, r=2, alpha=2)

print("=== LoraLinear 参数检查 ===")
print(f"  原始权重 requires_grad: {lora_layer.linear.weight.requires_grad}")
print(f"  lora_A requires_grad: {lora_layer.lora_A.requires_grad}")
print(f"  lora_B requires_grad: {lora_layer.lora_B.requires_grad}")

# 统计可训练参数
total = sum(p.numel() for p in lora_layer.parameters())
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
print(f"\n  总参数: {total}, 可训练: {trainable} ({100*trainable/total:.1f}%)")

# 前向传播测试
x = torch.tensor([[1.0, 0.5, -0.5, -1.0]])
y = lora_layer(x)
print(f"\n  输入 x: {x.tolist()}")
print(f"  输出 y: {y.tolist()}")

### 5. 玩具验证：LoRA 能学到和全量微调差不多的解吗？

造一个简单的回归任务：用 30 个数据点，学一个 `4 → 4` 的线性映射。
对比三种方案：
- **全量微调**：更新所有 16 个参数
- **LoRA (r=2)**：只更新 12 个参数
- **不训练**：baseline

In [ ]:
# ============================================================
# 玩具任务：学一个线性映射
# ============================================================
torch.manual_seed(42)

# 生成数据：学 y = X @ W_target
d = 4
W_target = torch.tensor([
    [ 0.5, -0.3,  0.8, -0.2],
    [-0.4,  0.6, -0.1,  0.7],
    [ 0.3, -0.5, -0.6,  0.4],
    [-0.7,  0.2,  0.5, -0.8],
])

N = 30
X_train = torch.randn(N, d)
y_train = X_train @ W_target + 0.05 * torch.randn(N, d)  # 加一点点噪声

print(f"训练数据: X ({N}x{d}), y ({N}x{d})")
print(f"目标映射 W_target:\n{W_target}")

# ============================================================
# 方案一：全量微调
# ============================================================
torch.manual_seed(100)
W_full = nn.Linear(d, d, bias=False)
W_full.weight.data = torch.randn(d, d)  # 随机初始化（模拟预训练权重）

print(f"\n=== 方案一：全量微调 ===")
print(f"初始权重:\n{W_full.weight.data}")
print(f"可训练参数: {sum(p.numel() for p in W_full.parameters())}")

opt_full = torch.optim.Adam(W_full.parameters(), lr=0.01)
losses_full = []
for epoch in range(200):
    opt_full.zero_grad()
    loss = F.mse_loss(W_full(X_train), y_train)
    loss.backward()
    opt_full.step()
    losses_full.append(loss.item())

print(f"  最终 loss: {losses_full[-1]:.6f}")

# ============================================================
# 方案二：LoRA (r=2)
# ============================================================
torch.manual_seed(100)  # 相同的初始权重
W_base = nn.Linear(d, d, bias=False)
W_base.weight.data = torch.randn(d, d)  # 和全量微调一样的初始化

# 包装成 LoRA
lora_model = LoraLinear(W_base, r=2, alpha=4)

print(f"\n=== 方案二：LoRA (r=2) ===")
print(f"原始权重（冻结）:\n{lora_model.linear.weight.data}")
print(f"可训练参数: {sum(p.numel() for p in lora_model.parameters() if p.requires_grad)}")

opt_lora = torch.optim.Adam(lora_model.parameters(), lr=0.01)
losses_lora = []
for epoch in range(200):
    opt_lora.zero_grad()
    loss = F.mse_loss(lora_model(X_train), y_train)
    loss.backward()
    opt_lora.step()
    losses_lora.append(loss.item())

print(f"  最终 loss: {losses_lora[-1]:.6f}")

# ============================================================
# 对比
# ============================================================
print(f"\n=== 对比 ===")
print(f"  全量微调 loss: {losses_full[-1]:.6f} (更新 {d*d} 个参数)")
print(f"  LoRA r=2  loss: {losses_lora[-1]:.6f} (更新 {2*d*2} 个参数)")

# 检查 LoRA 学到的权重是否接近目标
with torch.no_grad():
    learned_W = lora_model.weight  # 合并后的等效权重
    print(f"\n  LoRA 学到的等效权重:\n{learned_W}")
    print(f"  目标权重:\n{W_target}")
    print(f"  等效权重 vs 目标的 MSE: {F.mse_loss(learned_W, W_target):.6f}")

    full_W = W_full.weight.data
    print(f"  全量微调权重 vs 目标的 MSE: {F.mse_loss(full_W, W_target):.6f}")

### 6. 接入 MiniGPT：给 Attention 装 LoRA

回顾 MiniGPT 的 attention 层：
```
Q = W_Q @ X
K = W_K @ X
V = W_V @ X
O = W_O @ concat(heads)
```

我们给 Q 和 V 的投影层装 LoRA（最常见的做法），其他层冻结。

In [ ]:
# ============================================================
# 复用 Part 4 的 MiniGPT
# ============================================================
def get_sinusoidal_encoding(seq_len, d_model):
    position = torch.arange(seq_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer('pos_encoding', get_sinusoidal_encoding(max_len, d_model))
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, mask=None):
        B, S = x.shape
        x_emb = self.token_embedding(x) + self.pos_encoding[:S, :]
        for block in self.blocks:
            x_emb = block(x_emb, mask)
        x_emb = self.norm_final(x_emb)
        return self.lm_head(x_emb)

# 创建因果 mask
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

print("MiniGPT 加载完毕")

In [ ]:
# ============================================================
# 给 MiniGPT 的 Attention 装 LoRA
# ============================================================
def apply_lora_to_attention(model, r=16, alpha=32):
    """给模型所有 attention 的 W_Q 和 W_V 装上 LoRA"""
    lora_params = 0
    for block in model.blocks:
        attn = block.attention
        # 替换 W_Q
        attn.W_Q = LoraLinear(attn.W_Q, r=r, alpha=alpha)
        # 替换 W_V
        attn.W_V = LoraLinear(attn.W_V, r=r, alpha=alpha)
        # 统计
        lora_params += attn.W_Q.lora_A.numel() + attn.W_Q.lora_B.numel()
        lora_params += attn.W_V.lora_A.numel() + attn.W_V.lora_B.numel()
    return lora_params

VOCAB_SIZE = 50
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)

# 统计原始可训练参数
total_params = sum(p.numel() for p in model.parameters())
print(f"原始 MiniGPT 总参数: {total_params:,}")

# 装 LoRA
lora_params = apply_lora_to_attention(model, r=8, alpha=16)

# 重新统计
frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n装了 LoRA 之后:")
print(f"  冻结的参数: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")
print(f"  可训练参数(LoRA): {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"  仅用 {100*trainable_params/total_params:.2f}% 的参数！")

# 验证：原始权重确实冻结了
for name, param in model.named_parameters():
    if 'lora' not in name and 'lm_head' not in name and 'token_embedding' not in name:
        if param.requires_grad:
            print(f"  WARNING: {name} should be frozen but requires_grad=True")
print(f"\n原始 transformer 权重已全部冻结")

### 7. 训练演示：用 LoRA 做 SFT

用少量对话数据演示 LoRA 微调。目标：证明 LoRA 确实能学到东西，loss 下降。

In [ ]:
# ============================================================
# 准备 SFT 训练数据
# ============================================================
# 用简单的词表模拟对话
# 词表（50 个 token）——只用了其中一部分
word_to_id = {
    "BOS": 0, "EOS": 1,
    "你好": 2, "谢谢": 3, "再见": 4, "今天": 5, "天气": 6, "不错": 7,
    "你": 8, "是": 9, "谁": 10, "我": 11, "助手": 12, "AI": 13,
    "什么": 14, "为什么": 15, "怎么": 16, "做": 17, "可以": 18,
    "好": 19, "坏": 20, "喜欢": 21, "讨厌": 22, "知道": 23, "不知道": 24,
    "数学": 25, "英语": 26, "编程": 27, "写": 28, "代码": 29,
    "今天": 5, "星期": 30, "一": 31, "二": 32, "三": 33,
}

# 清理重复（今天重复了）
word_to_id = {
    "BOS": 0, "EOS": 1, "PAD": 0,
    "你好": 2, "谢谢": 3, "再见": 4, "今天": 5, "天气": 6, "不错": 7,
    "你": 8, "是": 9, "谁": 10, "我": 11, "助手": 12, "AI": 13,
    "什么": 14, "为什么": 15, "怎么": 16, "做": 17, "可以": 18,
    "好": 19, "坏": 20, "喜欢": 21, "讨厌": 22, "知道": 23, "不知道": 24,
    "数学": 25, "英语": 26, "编程": 27, "写": 28, "代码": 29,
    "星期": 30, "一": 31, "二": 32, "三": 33, "四": 34,
    "五": 35, "六": 36, "天": 37, "吗": 38, "的": 39,
    "很": 40, "不": 41, "对": 42, "错": 43, "嗯": 44,
    "啊": 45, "吧": 46, "会": 47, "了": 48, "在": 49,
}

# 生成训练数据：用户问题 + 助手回答
conversations = [
    ("你好", "你好！有什么可以帮你的吗？"),
    ("今天天气不错", "是的，今天天气很好，适合出去玩。"),
    ("你是谁", "我是一个 AI 助手，可以回答你的问题。"),
    ("你会写代码吗", "是的，我会写代码，尤其是 Python。"),
    ("谢谢", "不客气！还有什么想知道的吗？"),
    ("再见", "再见！祝你有美好的一天。"),
]

def encode_text(text):
    """简单的逐词编码（中文按词分）"""
    ids = []
    for word in text:
        if word in word_to_id:
            ids.append(word_to_id[word])
        else:
            ids.append(0)  # 未知词用 PAD/BOS
    return ids

print("训练数据示例:")
for q, a in conversations[:3]:
    print(f"  User: {q}  →  Assistant: {a}")
    q_ids = encode_text(q)
    a_ids = encode_text(a)
    print(f"    IDs: User {q_ids}, Assistant {a_ids}")

In [ ]:
# ============================================================
# LoRA SFT 训练循环
# ============================================================
VOCAB_SIZE = 50
SEQ_LEN = 24
torch.manual_seed(42)

# 创建模型并装 LoRA
model = MiniGPT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2, max_len=SEQ_LEN)
apply_lora_to_attention(model, r=8, alpha=16)

# 只优化可训练参数（LoRA 的 A 和 B + lm_head + embedding）
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.001
)

print(f"优化器管理的参数数量: {sum(p.numel() for p in optimizer.param_groups[0]['params']):,}")

# ============================================================
# 训练
# ============================================================
NUM_EPOCHS = 50
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0

    for user_text, assistant_text in conversations:
        # 构造完整的输入序列: [BOS] + user + assistant + [EOS]
        user_ids = encode_text(user_text)
        asst_ids = encode_text(assistant_text)
        full_seq = [word_to_id["BOS"]] + user_ids + asst_ids + [word_to_id["EOS"]]

        # Padding 到固定长度
        if len(full_seq) > SEQ_LEN:
            full_seq = full_seq[:SEQ_LEN]
        input_ids = torch.tensor([full_seq + [0] * (SEQ_LEN - len(full_seq))])

        # 前向传播
        mask = causal_mask(SEQ_LEN)
        logits = model(input_ids, mask)  # (1, S, V)

        # Labels：右移一位
        labels = torch.cat([input_ids[:, 1:], torch.zeros(1, 1, dtype=torch.long)], dim=1)

        # Loss：只在非 PAD 位置计算
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            labels.view(-1),
            ignore_index=0  # PAD
        )

        epoch_loss += loss.item()

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = epoch_loss / len(conversations)
    losses.append(avg_loss)

    if epoch % 5 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.4f}")

print(f"\n初始 Loss: {losses[0]:.4f} → 最终 Loss: {losses[-1]:.4f}")
print(f"Loss 下降: {losses[0] - losses[-1]:.4f}")

In [ ]:
# ============================================================
# 可视化 Loss 下降
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.plot(losses, 'b-', linewidth=1.5)
plt.axhline(y=losses[0], color='gray', linestyle='--', alpha=0.5, label=f'初始: {losses[0]:.2f}')
plt.axhline(y=losses[-1], color='red', linestyle='--', alpha=0.5, label=f'最终: {losses[-1]:.2f}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LoRA 微调 MiniGPT —— Loss 下降曲线')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n结论：只更新了不到 1% 的参数，Loss 依然显著下降！")
print("这就是 LoRA 的魔力 —— 用极少的参数实现接近全量微调的效果。")

### 8. 推理时：合并权重，零额外开销

LoRA 的另一个巧妙之处：训练完成后，可以把 `W + (alpha/r)*AB` **预先算好**，
推理时和普通模型一模一样，没有任何额外计算。

In [ ]:
# ============================================================
# 合并 LoRA 权重到原始权重
# ============================================================
def merge_lora(model):
    """把 LoRA 权重合并进原始权重，推理时零额外开销"""
    for block in model.blocks:
        attn = block.attention
        for name in ['W_Q', 'W_V']:
            layer = getattr(attn, name)
            if isinstance(layer, LoraLinear):
                # 计算合并后的权重
                merged_weight = layer.linear.weight.data + layer.lora_A.data @ layer.lora_B.data * layer.scaling
                # 替换回普通 nn.Linear
                new_linear = nn.Linear(layer.in_features, layer.out_features, bias=False)
                new_linear.weight.data = merged_weight
                setattr(attn, name, new_linear)

# 合并前后对比
trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"合并前可训练参数: {trainable_before}")

merge_lora(model)

trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"合并后可训练参数: {trainable_after}")
print(f"\n合并后推理速度和普通模型完全一样，没有任何 LoRA 开销！")

### 9. 实践要点总结

| 问题 | 答案 |
|------|------|
| LoRA 加在哪些层？ | 最常见：Q 和 V 的投影层。也可以加在 FFN、lm_head |
| 秩 r 选多大？ | r=8 或 16 通常足够。r 太小学不到东西，太大接近全量微调 |
| alpha 怎么设？ | 通常是 r 的 2-4 倍。alpha/r 是实际的缩放因子 |
| 一个基座模型可以有几个 LoRA？ | 理论上无限个。每个 LoRA 只有几 MB，可以随时切换 |
| 和全量微调比效果差多少？ | 通常差距很小（<2%）。数据少时 LoRA 甚至更好（不容易过拟合）|
| QLoRA 是什么？ | LoRA + 4-bit 量化。用更少的显存，效果几乎不变 |

**一句话总结**：LoRA = 冻结原始模型，只在旁路训练两个小矩阵 A 和 B。
训练成本降到原来的 1%，效果几乎不变。训练完可以把 A×B 合并进原始权重，推理零开销。